# MM4250: analysis

Everything here works on saved data: **no instruments**, so it runs while someone else has the switch and the VNA, or on a laptop with a copy of the database or the files.

1. **What's saved**: takes, the newest runs, folders of Touchstone files.
2. **Calibrate raw data**: (a) a take in the database, with whichever NIST definitions you choose; (b) raw Touchstone files (a take's or a stuck check's folder, an export, old files), using the switch's own open/short/load among them.
3. **Plot**: from the database or from files, |S| beside a Smith chart, one figure per trace or all together.
4. **Resonance quick look**: where the dip is, how deep, rough width and Q_L, over- or undercoupled. No fit.
5. **Temperatures**: MXP, MXP2, STP before and after every position of a take or check.
6. **Drift and repeatability** of a take.
7. **Export** runs to Touchstone files.

Run Setup first; after that each section stands on its own. Nothing is saved unless a `SAVE...` option says so, and nothing ever overwrites a file.

**Setup**: which cooldown (for the database's location) and which NIST definitions to calibrate with.

In [ ]:
COOLDOWN_FILE = "cooldowns/CD05.json"   # this cooldown's file (see README.md)
DEFINITIONS = "3K"                      # NIST's: "3K" cold (holds to 25 mK), "295K" warm; or a folder path

import sys
from pathlib import Path
try:
    here = Path(_dh[0])                      # this notebook's folder (Jupyter)
except NameError:
    here = Path.cwd()
sys.path[:0] = [str(here)]
%matplotlib inline

import db, ecal, plots, analysis
cfg = db.load_cooldown(here / COOLDOWN_FILE)
DB = db.open_database(cfg, "vna", create=False)          # takes
STUCK_DB = db.open_database(cfg, "stuck", create=False)  # stuck and thru checks
FILES = DB.parent / "Sweeps"                             # saved Touchstone files
print(f"takes:  {DB}{'' if DB.is_file() else '   (not made yet)'}")
print(f"checks: {STUCK_DB}{'' if STUCK_DB.is_file() else '   (not made yet)'}")
print(f"files:  {FILES}")

## 1. What's saved

In [ ]:
db.list_takes(DB)
print()
analysis.list_runs(DB, last=20)
print()
analysis.list_folders(FILES);

## 2a. Calibrate a take from the database

For a take saved raw only (no fridge reading, or between 4.5 and 250 K), or to see a take with other definitions. `SAVE = True` adds the result to the database as corrected runs, linked to the raw ones (`definitions_chosen` = by hand); a take that already has runs with the same definitions is left alone.

In [ ]:
TAKE_ID = None           # None = the newest take; or one from the list above
SAVE = False             # True: save the result as corrected runs in the database

take = db.load_take(TAKE_ID, DB)
cal_db = ecal.correct_take(take, DEFINITIONS)
plots.plot_mag_smith(analysis.traces_from_calibration(cal_db), together=True,
                     title=f"take {take['take_id']}, {take['temp_str']}, {cal_db['definitions']}")
if SAVE:
    db.save_corrected(DB, cal_db, take)

## 2b. Calibrate raw Touchstone files

A folder of raw `.s2p` files from one session: a take's folder (`Sweeps/takes/...`), a stuck check's (`Sweeps/stuck_check/...`, also the basic stuck check's), an export, or old files. The position comes from each file name (`ALL_OPEN`, `INTERNAL_SHORT`, `INTERNAL_LOAD`, `RF1`...`RF6`); the switch's own open, short and load among them do the calibration, so they must be from the same session as the channels.

- `SPARAM`: `"S21"` in the fridge (circulator wiring), `"S11"` on the bench.
- `STANDARDS`: several files of one standard (before/after sets, a stuck check's ALL_OPENs) are averaged (`"mean"`), or use the `"first"` / `"last"`. After a stuck-closed channel the later ALL_OPENs aren't opens: use `"first"`.
- `SAVE_FILES = True`: one `RF<n>_calibrated_<definitions>.s1p` per channel in a `calibrated/` folder inside `FOLDER`.

In [ ]:
FOLDER = None            # None = the newest folder of sweeps for this cooldown; or a path
SPARAM = "S21"
CHANNELS = None          # None = every RF<n> file; or e.g. [4]
STANDARDS = "mean"       # "mean", "first" or "last"
SAVE_FILES = False

folder = Path(FOLDER) if FOLDER else analysis.newest_folder(FILES)
if folder is None:
    print(f"No folders of sweeps under {FILES}: set FOLDER")
else:
    files = analysis.load_folder(folder)
    cal_files = analysis.calibrate_files(files, DEFINITIONS, sparam=SPARAM, channels=CHANNELS,
                                         standards=STANDARDS)
    plots.plot_mag_smith(analysis.traces_from_calibration(cal_files))
    if SAVE_FILES:
        analysis.save_calibrated(cal_files, folder / "calibrated")

## 3. Plot from the database or from files

Pick one source: a take (`TAKE_ID`; `WHICH` = `"corrected"` or `"raw"`), a list of run ids (`RUN_IDS`), or Touchstone files (`PATHS`: a list of files, already calibrated or not). `TOGETHER = True` overlays everything on one figure (e.g. the same channel from several takes: list their corrected run ids). `SAVE_FIGURES`: a folder for the PNGs.

In [ ]:
TAKE_ID = None           # None = the newest take (used when RUN_IDS and PATHS are empty)
WHICH = "corrected"      # "corrected" or "raw"
RUN_IDS = []             # e.g. [12, 40]: plot these runs instead
PATHS = []               # e.g. [FILES / "takes" / "<folder>" / "RF4_corrected_rep1_run12.s1p"]
TOGETHER = False
SAVE_FIGURES = None      # e.g. FILES / "figures"

if PATHS:
    traces = analysis.traces_from_files(PATHS)
elif RUN_IDS:
    traces = analysis.traces_from_runs(db.load_runs(RUN_IDS, DB))
else:
    traces = analysis.traces_from_take(db.load_take(TAKE_ID, DB), which=WHICH)
plots.plot_mag_smith(traces, together=TOGETHER, save=SAVE_FIGURES);

## 4. Resonance quick look

Fit-free numbers for one corrected trace: the dip's frequency and depth, a rough half-depth width and Q_L, and whether it looks over- or undercoupled (an overcoupled resonance swings round the centre of the Smith chart; at high Q its |G| can stay near 0 dB, so look at the phase swing too). Zoom in with `START_MHZ` / `STOP_MHZ` once you know where it is. This is a first look, not a Q fit.

In [ ]:
TAKE_ID = None           # None = the newest take
CHANNEL = 4              # the resonator's switch channel
START_MHZ, STOP_MHZ = None, None   # e.g. 540, 560

trace = next((t for t in analysis.traces_from_take(db.load_take(TAKE_ID, DB)) if t["channel"] == CHANNEL), None)
if trace is None:
    print(f"No corrected RF{CHANNEL} in that take")
else:
    trace = analysis.zoom(trace, START_MHZ, STOP_MHZ)
    res = analysis.resonance(trace)
    analysis.print_resonance(res)
    plots.plot_mag_smith([trace], mark_hz=res["f_min_hz"]);

## 5. Temperatures during a take or a check

MXP, MXP2 and STP just before (dot) and just after (x) each position. A take from the takes database, or a check (`CHECK_ID`, from the stuck-check database).

In [ ]:
TAKE_ID = None           # None = the newest take
CHECK_ID = None          # or a stuck/thru check's id (then TAKE_ID is ignored)

rows = (analysis.temperature_log(STUCK_DB, check_id=CHECK_ID) if CHECK_ID
        else analysis.temperature_log(DB, take_id=TAKE_ID))
plots.plot_temperatures(rows);

## 6. Drift and repeatability of a take

Drift: how far the standards moved between the before and after sets (compare with NIST's few 1e-2). Repeatability: how far each repeat's corrected trace is from their mean (needs `repeats` > 1 in the cooldown file).

In [ ]:
TAKE_ID = None           # None = the newest take

take = db.load_take(TAKE_ID, DB)
ecal.drift(take)
analysis.repeat_spread(take);

## 7. Export to Touchstone

Runs from the database as `.s2p` (raw) / `.s1p` (corrected) files in `Sweeps/export/` beside the database. Never overwrites.

In [ ]:
TAKE_ID = None           # None = the newest take (all its runs), unless RUN_IDS is set
RUN_IDS = []             # e.g. [12, 13]
EXPORT = False           # True: write the files

ids = RUN_IDS or db.take_run_ids(db.load_take(TAKE_ID, DB))
print(f"{len(ids)} run(s): {ids[:10]}{' ...' if len(ids) > 10 else ''}")
if EXPORT:
    db.export_touchstone(ids, DB)